# Dataset Statistics — CLEF 2026 SimpleText Task 1
## Tokatrons Team

Computes training set statistics for §4 of the paper.
Run this after mounting Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'

import pandas as pd
import ast

train_path = f'{DATA_DIR}/cochraneauto_sents_train.csv'
val_path   = f'{DATA_DIR}/cochraneauto_sents_val.csv'

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)

print(f'Training pairs: {len(train_df)}')
print(f'Validation pairs: {len(val_df)}')
print(f'Columns: {list(train_df.columns)}')

In [ ]:
# Parse 'simple' column (stringified list) into plain text
def parse_simple(s):
    try:
        lst = ast.literal_eval(s)
        return ' '.join(lst) if isinstance(lst, list) else str(lst)
    except:
        return str(s)

train_df['simple_text'] = train_df['simple'].apply(parse_simple)
print('Sample parsed target:', repr(train_df['simple_text'].iloc[0][:120]))

In [ ]:
# Source sentence statistics
train_df['src_len_tokens'] = train_df['complex'].apply(lambda x: len(x.split()))
train_df['src_len_chars']  = train_df['complex'].apply(lambda x: len(x))

print('=== SOURCE SENTENCE LENGTHS ===')
print(f'Tokens - Mean: {train_df["src_len_tokens"].mean():.1f}, Median: {train_df["src_len_tokens"].median():.0f}')
print(f'Tokens - Min: {train_df["src_len_tokens"].min()}, Max: {train_df["src_len_tokens"].max()}')
print(f'Tokens - Std: {train_df["src_len_tokens"].std():.1f}')
print(f'Chars  - Mean: {train_df["src_len_chars"].mean():.1f}, Median: {train_df["src_len_chars"].median():.0f}')

In [ ]:
# Target sentence statistics
train_df['tgt_len_tokens'] = train_df['simple_text'].apply(lambda x: len(x.split()))
train_df['tgt_len_chars']  = train_df['simple_text'].apply(lambda x: len(x))

print('=== TARGET SENTENCE LENGTHS ===')
print(f'Tokens - Mean: {train_df["tgt_len_tokens"].mean():.1f}, Median: {train_df["tgt_len_tokens"].median():.0f}')
print(f'Tokens - Min: {train_df["tgt_len_tokens"].min()}, Max: {train_df["tgt_len_tokens"].max()}')
print(f'Tokens - Std: {train_df["tgt_len_tokens"].std():.1f}')
print(f'Chars  - Mean: {train_df["tgt_len_chars"].mean():.1f}, Median: {train_df["tgt_len_chars"].median():.0f}')
print()
print('=== COMPRESSION RATIO (source_tokens / target_tokens) ===')
train_df['compression'] = train_df['src_len_tokens'] / train_df['tgt_len_tokens'].clip(lower=1)
print(f'Mean: {train_df["compression"].mean():.2f}, Median: {train_df["compression"].median():.2f}')

In [ ]:
# Label distribution
print('=== LABEL DISTRIBUTION ===')
label_counts = train_df['label'].value_counts()
for label, count in label_counts.items():
    print(f'{label:>10}: {count:>5} ({100*count/len(train_df):.1f}%)')

In [ ]:
# Check for empty targets
empty_targets = (train_df['simple_text'].str.strip() == '').sum()
print(f'Empty targets: {empty_targets} ({100*empty_targets/len(train_df):.1f}%)')

In [ ]:
# Vocabulary analysis
from collections import Counter

all_src_words = ' '.join(train_df['complex']).lower().split()
all_tgt_words = ' '.join(train_df['simple_text']).lower().split()

src_vocab = set(all_src_words)
tgt_vocab = set(all_tgt_words)

print('=== VOCABULARY ===')
print(f'Source unique tokens: {len(src_vocab)}')
print(f'Target unique tokens: {len(tgt_vocab)}')
print(f'Unique tokens in target but not source: {len(tgt_vocab - src_vocab)}')
print(f'Type-token ratio (source): {len(src_vocab)/len(all_src_words):.4f}')
print(f'Type-token ratio (target): {len(tgt_vocab)/len(all_tgt_words):.4f}')

In [ ]:
# Export results for copying into paper
print('=' * 60)
print('SUMMARY FOR TABLE IN PAPER')
print('=' * 60)
print(f'Training pairs & {len(train_df)} \\\\')
print(f'Source tokens (mean $\\pm$ std) & {train_df["src_len_tokens"].mean():.0f} $\\pm$ {train_df["src_len_tokens"].std():.0f} \\\\')
print(f'Target tokens (mean $\\pm$ std) & {train_df["tgt_len_tokens"].mean():.0f} $\\pm$ {train_df["tgt_len_tokens"].std():.0f} \\\\')
print(f'Source vocabulary size & {len(src_vocab):,} \\\\')
print(f'Target vocabulary size & {len(tgt_vocab):,} \\\\')
print(f'Compression ratio (mean) & {train_df["compression"].mean():.2f} \\\\')
print(f'Empty targets & {empty_targets} ({100*empty_targets/len(train_df):.1f}\\%) \\\\')
print(f'Rephrase & {label_counts.get("rephrase", 0):,} ({100*label_counts.get("rephrase", 0)/len(train_df):.1f}\\%) \\\\')
print(f'Delete & {label_counts.get("delete", 0):,} ({100*label_counts.get("delete", 0)/len(train_df):.1f}\\%) \\\\')
print(f'Copy & {label_counts.get("copy", 0):,} ({100*label_counts.get("copy", 0)/len(train_df):.1f}\\%) \\\\')
print(f'Merge & {label_counts.get("merge", 0):,} ({100*label_counts.get("merge", 0)/len(train_df):.1f}\\%) \\\\')
print(f'Split & {label_counts.get("split", 0):,} ({100*label_counts.get("split", 0)/len(train_df):.1f}\\%) \\\\')